---
title: Performance comparison
---

# Aggregating ICESat-2 elevations from the cloud

[magg](https://github.com/englacial/magg) aggregates ICESat-2 ATL06 data
into gridded products using 1,700+ parallel Lambda workers. Each worker
reads hundreds of HDF5 granules for a single spatial cell: all 6 beam
groups per granule, with quality filtering and weighted-mean elevation
statistics.

This notebook reproduces that workflow on a realistic subset: search for
ATL06 granules over a region of Antarctica, read `h_li`, `h_li_sigma`,
`latitude`, `longitude`, and `atl06_quality_summary` from all 6 beams,
filter by quality, and compute weighted-mean elevation in 1-degree
latitude bins. We compare two approaches:

1. **h5coro** (sequential) -- how magg reads data today
2. **async-hdf5 + zarrs** (concurrent) -- Rust I/O with zarrs-python's Rust codec pipeline

In [ ]:
import warnings

warnings.filterwarnings("ignore", message="Numcodecs codecs are not in the Zarr")
warnings.filterwarnings(
    "ignore", category=UserWarning, message=".*does not have a Zarr V3 specification.*"
)
warnings.filterwarnings("ignore", category=FutureWarning, module="earthaccess")
warnings.filterwarnings("ignore", category=DeprecationWarning, module="tqdm")

## Find granules

We search for ATL06 granules over the Amundsen Sea sector of West
Antarctica, one of the most data-dense regions in the ICESat-2 archive.
A real magg worker processes 300-400 granules for a single cell near
the poles; here we use a smaller subset to keep the demo fast.

In [ ]:
import asyncio
import time
from urllib.parse import urlparse

import earthaccess
import h5coro
import numpy as np
import zarr
from async_hdf5.store import HTTPStore
from async_hdf5.zarr import open_hdf5
from h5coro import webdriver

auth = earthaccess.login()

results = earthaccess.search_data(
    short_name="ATL06",
    cloud_hosted=True,
    bounding_box=(-120, -76, -95, -73),  # Amundsen Sea, West Antarctica
    temporal=("2024-01-01", "2024-03-31"),
    count=20,
)

urls = []
for granule in results:
    links = granule.data_links()
    h5 = [l for l in links if l.endswith(".h5")]
    if h5:
        urls.append(h5[0])

token = auth.token["access_token"]
print(f"{len(urls)} granules over Amundsen Sea sector")

## magg's data access pattern

Each granule contains 6 beam groups (`gt1l` through `gt3r`). For each
beam, magg reads:

- `h_li` -- land ice surface elevation
- `h_li_sigma` -- elevation uncertainty (used for weighted averaging)
- `latitude`, `longitude` -- coordinates
- `atl06_quality_summary` -- quality flag (0 = good)

This is defined in magg's
[`atl06.yaml`](https://github.com/englacial/magg/blob/main/src/magg/configs/atl06.yaml)
config.

In [ ]:
BEAMS = ["gt1l", "gt1r", "gt2l", "gt2r", "gt3l", "gt3r"]


def make_store(url):
    """Create an authenticated HTTPStore for a given URL."""
    parsed = urlparse(url)
    dir_path, _, filename = parsed.path.rpartition("/")
    base_url = f"{parsed.scheme}://{parsed.netloc}{dir_path}"
    store = HTTPStore.from_url(
        base_url,
        client_options={"default_headers": {"Authorization": f"Bearer {token}"}},
    )
    return store, filename


def is_valid(h_li, sigma, lat, quality):
    """magg's quality + validity filter for ATL06."""
    return (
        (quality == 0)
        & np.isfinite(h_li)
        & (h_li < 1e10)
        & np.isfinite(sigma)
        & (sigma > 0)
        & np.isfinite(lat)
    )

## Helper: weighted-mean aggregation by latitude

magg computes inverse-variance weighted mean elevation per spatial cell.
Here we bin by 1-degree latitude bands instead of HEALPix cells, but the
aggregation logic is the same: $\bar{h} = \sum w_i h_i / \sum w_i$ where
$w_i = 1 / \sigma_i^2$.

In [ ]:
def aggregate(all_lat, all_h_li, all_sigma):
    """Weighted-mean elevation in 1-degree latitude bands."""
    lat_bins = np.arange(-80, -70, 1.0)
    bin_idx = np.digitize(all_lat, lat_bins) - 1
    means = np.full(len(lat_bins) - 1, np.nan)
    counts = np.zeros(len(lat_bins) - 1, dtype=int)
    for i in range(len(lat_bins) - 1):
        mask = bin_idx == i
        if mask.any():
            w = 1.0 / all_sigma[mask] ** 2
            means[i] = np.average(all_h_li[mask], weights=w)
            counts[i] = mask.sum()
    return lat_bins, means, counts

## 1. The baseline: h5coro (sequential)

This is how magg reads data today: one file at a time, looping over
every granule. Each granule reads all 6 beams.

In [ ]:
h5coro_credentials = {"bearer_token": token}

t0 = time.perf_counter()
all_lat_h5coro = []
all_hli_h5coro = []
all_sigma_h5coro = []

for u in urls:
    h5obj = h5coro.H5Coro(u, webdriver.HTTPDriver, credentials=h5coro_credentials)

    for beam in BEAMS:
        prefix = f"/{beam}/land_ice_segments"
        datasets = [
            f"{prefix}/h_li",
            f"{prefix}/h_li_sigma",
            f"{prefix}/latitude",
            f"{prefix}/longitude",
            f"{prefix}/atl06_quality_summary",
        ]
        try:
            result = h5obj.readDatasets(datasets=datasets, block=True)
        except Exception:
            continue

        h_li = np.asarray(result[f"{prefix}/h_li"])
        sigma = np.asarray(result[f"{prefix}/h_li_sigma"])
        lat = np.asarray(result[f"{prefix}/latitude"])
        quality = np.asarray(result[f"{prefix}/atl06_quality_summary"])

        valid = is_valid(h_li, sigma, lat, quality)
        all_lat_h5coro.append(lat[valid])
        all_hli_h5coro.append(h_li[valid])
        all_sigma_h5coro.append(sigma[valid])

    h5obj.close()

all_lat_h5coro = np.concatenate(all_lat_h5coro)
all_hli_h5coro = np.concatenate(all_hli_h5coro)
all_sigma_h5coro = np.concatenate(all_sigma_h5coro)
lat_bins, means_h5coro, counts_h5coro = aggregate(
    all_lat_h5coro, all_hli_h5coro, all_sigma_h5coro
)

elapsed_h5coro = time.perf_counter() - t0
print(
    f"h5coro sequential: {elapsed_h5coro:.2f}s for {len(urls)} granules x {len(BEAMS)} beams"
)
print(f"Total valid points: {len(all_lat_h5coro):,}")

## 2. async-hdf5 + zarrs (concurrent)

async-hdf5 exposes each HDF5 group as a Zarr v3 store. We read all
granules x beams concurrently with `asyncio.gather`.
[zarrs-python](https://github.com/ilan-gold/zarrs-python) replaces
zarr-python's codec pipeline with one backed by the
[zarrs](https://docs.rs/zarrs) Rust crate, so decompression also
runs in Rust threads with no GIL.

In [ ]:
zarr.config.set(
    {
        "async.concurrency": 128,
        "codec_pipeline.path": "zarrs.ZarrsCodecPipeline",
    }
)


async def read_beam(url, beam):
    """Read one beam from one granule; return valid (lat, h_li, sigma)."""
    store, filename = make_store(url)
    try:
        hdf5_store = await open_hdf5(
            path=filename,
            store=store,
            group=f"{beam}/land_ice_segments",
        )
    except Exception:
        return None
    group = zarr.open_group(hdf5_store, mode="r")
    h_li = group["h_li"][:]
    sigma = group["h_li_sigma"][:]
    lat = group["latitude"][:]
    quality = group["atl06_quality_summary"][:]

    valid = is_valid(h_li, sigma, lat, quality)
    if not valid.any():
        return None
    return lat[valid], h_li[valid], sigma[valid]


t0 = time.perf_counter()
tasks = [read_beam(u, b) for u in urls for b in BEAMS]
results_async = await asyncio.gather(*tasks)
results_async = [r for r in results_async if r is not None]

all_lat_async = np.concatenate([r[0] for r in results_async])
all_hli_async = np.concatenate([r[1] for r in results_async])
all_sigma_async = np.concatenate([r[2] for r in results_async])
lat_bins, means_async, counts_async = aggregate(
    all_lat_async, all_hli_async, all_sigma_async
)

elapsed_async = time.perf_counter() - t0
print(
    f"async-hdf5 + zarrs: {elapsed_async:.2f}s for {len(urls)} granules x {len(BEAMS)} beams"
)
print(f"Total valid points: {len(all_lat_async):,}")

## Results

In [ ]:
import pandas as pd

n_reads = len(urls) * len(BEAMS)
speedup = elapsed_h5coro / elapsed_async

summary = pd.DataFrame(
    {
        "Library": [
            "h5coro (sequential)",
            "async-hdf5 + zarrs (concurrent)",
        ],
        f"Time ({n_reads} beam reads)": [
            f"{elapsed_h5coro:.1f}s",
            f"{elapsed_async:.1f}s",
        ],
        "Valid points": [
            f"{len(all_lat_h5coro):,}",
            f"{len(all_lat_async):,}",
        ],
        "Speedup": ["baseline", f"{speedup:.1f}x"],
    }
)
summary = summary.set_index("Library")
summary

## Aggregated elevation profile

Weighted-mean ice sheet elevation by 1-degree latitude band over the
Amundsen Sea sector. Both approaches produce the same result, the
difference is how long the I/O and decompression take.

In [ ]:
import matplotlib.pyplot as plt

bin_centers = (lat_bins[:-1] + lat_bins[1:]) / 2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Elevation profile
valid_bins = ~np.isnan(means_async)
ax1.plot(bin_centers[valid_bins], means_h5coro[valid_bins], "o-", label="h5coro")
ax1.plot(
    bin_centers[valid_bins], means_async[valid_bins], "s--", label="async-hdf5 + zarrs"
)
ax1.set_xlabel("Latitude (deg)")
ax1.set_ylabel("Weighted mean elevation (m)")
ax1.set_title("Ice sheet elevation by latitude (Amundsen Sea)")
ax1.legend()

# Point counts per bin
ax2.bar(bin_centers[valid_bins], counts_async[valid_bins], width=0.8, alpha=0.7)
ax2.set_xlabel("Latitude (deg)")
ax2.set_ylabel("Number of valid points")
ax2.set_title("Data density by latitude")

fig.tight_layout()

## Timing

Where does the speedup come from?

- **Concurrent I/O**: all granule/beam combinations are fetched and
  parsed in parallel instead of sequentially (Rust async, no GIL).
- **Rust decompression**: zarrs decodes multiple chunks in parallel
  using Rust threads, bypassing the GIL.

A real magg worker processes 300-400 granules x 6 beams = 1,800-2,400
HDF5 group reads. async-hdf5 + zarrs gives you concurrency within a
single process, which means fewer Lambda workers doing more work each.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
libraries = ["h5coro\n(sequential)", "async-hdf5 + zarrs\n(concurrent)"]
times = [elapsed_h5coro, elapsed_async]
colors = ["#e74c3c", "#2ecc71"]
bars = ax.bar(
    libraries, times, color=colors, width=0.4, edgecolor="black", linewidth=0.5
)

for bar, t in zip(bars, times):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(times) * 0.02,
        f"{t:.1f}s",
        ha="center",
        fontweight="bold",
    )

ax.set_ylabel("Wall-clock time (s)")
ax.set_title(f"Aggregation time: {len(urls)} granules x {len(BEAMS)} beams")
ax.set_ylim(0, max(times) * 1.15)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()

The comparison is apples-to-oranges by design. h5coro **must** read
sequentially, it's not thread-safe
([englacial/magg#5](https://github.com/englacial/magg/issues/5)).
magg works around this by pushing parallelism to infrastructure
(1,700 Lambda workers). async-hdf5 + zarrs gives you concurrency
within a single process, and zarrs-python accelerates the
decompression step that remains after I/O is parallelized.